# Student Performance Prediction using Machine Learning

Predicting a student's final grade (`G3`) using academic, demographic, family, social, and school-related information.

**Final selected model:** Random Forest Regressor with 100 trees.

> **GitHub note:** This notebook is stored in `notebooks/`, so dataset and model paths use `../data/` and `../models/`.


## Before Running the Notebook

1. Put `student-mat.csv` inside the project's `data/` folder.
2. Open this notebook from the `notebooks/` folder.
3. Run the cells from top to bottom.


## 1. Problem Statement

The objective is to predict a student's final grade (`G3`) from the other information available in the dataset.

The project follows this workflow:

**Data → EDA → Preprocessing → Model Training → Cross-Validation → Model Comparison → Final Model → Evaluation → New Student Prediction**

## 2. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## 3. Load Dataset

The original notebook loads the UCI-style student mathematics dataset from `student-mat.csv`.

The dataset contains **395 rows and 33 columns**. The target variable is `G3`, so there are **32 input features** in the final model.

In [ ]:
df = pd.read_csv("../data/student-mat.csv", sep=";")
df.head()

In [ ]:
print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

## 4. Basic Dataset Inspection

In [ ]:
df.info()

In [ ]:
print("Missing values:")
print(df.isnull().sum())

In [ ]:
df.describe()

In [ ]:
categorical_cols = df.select_dtypes(include="object").columns
numerical_cols = df.select_dtypes(exclude="object").columns

print("Categorical columns:")
print(categorical_cols.tolist())

print("\nNumerical columns:")
print(numerical_cols.tolist())

## 5. Exploratory Data Analysis

EDA is used to understand the target variable and examine relationships between student characteristics and final performance.

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(df["G3"], bins=15, kde=True)
plt.xlabel("Final Grade (G3)")
plt.ylabel("Number of Students")
plt.title("Distribution of Final Grades")
plt.show()

print("Average Final Grade:", df["G3"].mean())
print("Minimum Final Grade:", df["G3"].min())
print("Maximum Final Grade:", df["G3"].max())

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(x="studytime", y="G3", data=df)
plt.xlabel("Study Time")
plt.ylabel("Final Grade (G3)")
plt.title("Study Time vs Final Grade")
plt.show()

In [ ]:
corr = df.corr(numeric_only=True)

print("Correlation with G3:")
display(corr["G3"].sort_values(ascending=False))

In [ ]:
plt.figure(figsize=(10, 7))
sns.heatmap(corr, annot=True, fmt=".2f")
plt.title("Correlation Matrix")
plt.show()

### Selected Categorical and Academic Relationships

In [ ]:
display(df.groupby("sex")["G3"].mean())
display(df.groupby("Medu")["G3"].mean())
display(df.groupby("higher")["G3"].mean())
display(df.groupby("failures")["G3"].mean())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))

sns.barplot(x="sex", y="G3", data=df, ax=axes[0, 0])
axes[0, 0].set_title("Gender vs Average Final Grade")

sns.barplot(x="Medu", y="G3", data=df, ax=axes[0, 1])
axes[0, 1].set_title("Mother's Education vs Average Final Grade")

sns.barplot(x="higher", y="G3", data=df, ax=axes[1, 0])
axes[1, 0].set_title("Higher Education Intention vs Final Grade")

sns.barplot(x="failures", y="G3", data=df, ax=axes[1, 1])
axes[1, 1].set_title("Past Failures vs Final Grade")

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

sns.regplot(x="absences", y="G3", data=df, ax=axes[0])
axes[0].set_title("Absences vs Final Grade")

sns.scatterplot(x="G1", y="G3", data=df, ax=axes[1])
axes[1].set_title("G1 vs G3")

sns.scatterplot(x="G2", y="G3", data=df, ax=axes[2])
axes[2].set_title("G2 vs G3")

plt.tight_layout()
plt.show()

print("Correlation: absences vs G3 =", df["absences"].corr(df["G3"]))
print("Correlation: G1 vs G3       =", df["G1"].corr(df["G3"]))
print("Correlation: G2 vs G3       =", df["G2"].corr(df["G3"]))

### EDA Observation

The earlier grades (`G1` and especially `G2`) show a strong relationship with the final grade `G3`. Absences also show a relationship with final performance.

This project intentionally keeps `G1` and `G2` as input features because the goal is to predict the final grade when earlier-period grades are already available.

## 6. Define Features and Target

`G3` is the target.

All remaining 32 columns are used as input features.

In [ ]:
X = df.drop(columns=["G3"])
y = df["G3"]

print("X shape:", X.shape)
print("y shape:", y.shape)

## 7. Train-Test Split

An 80/20 split is used with `random_state=42` so that the split is reproducible.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

## 8. Preprocessing

The dataset contains both categorical and numerical features.

- Categorical features → One-Hot Encoding
- Numerical features → passed through unchanged

`handle_unknown="ignore"` allows the model to handle an unseen category during prediction without failing.

The preprocessing is placed inside a Scikit-learn `Pipeline`, which keeps preprocessing and modeling together and helps avoid data leakage during cross-validation.

In [ ]:
categorical_cols = X.select_dtypes(include="object").columns.tolist()
numerical_cols = X.select_dtypes(exclude="object").columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ("num", "passthrough", numerical_cols)
    ]
)

## 9. Baseline Models

We compare five regression approaches:

1. Linear Regression
2. Decision Tree
3. Tuned Decision Tree
4. Tuned Random Forest
5. Random Forest with 100 trees

The final comparison uses the same 5-fold cross-validation procedure for every model.

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)

models = {
    "Linear Regression": LinearRegression(),

    "Decision Tree": DecisionTreeRegressor(
        random_state=42
    ),

    "Tuned Decision Tree": DecisionTreeRegressor(
        max_depth=3,
        random_state=42
    ),

    "Tuned Random Forest": RandomForestRegressor(
        n_estimators=500,
        max_depth=2,
        min_samples_split=30,
        min_samples_leaf=15,
        max_features=1.0,
        random_state=42
    ),

    "Random Forest (100 trees)": RandomForestRegressor(
        n_estimators=100,
        random_state=42
    )
}

## 10. Five-Fold Cross-Validation

For each model we calculate:

- **R²** — higher is better
- **MAE** — lower is better
- **RMSE** — lower is better

The preprocessing is fitted separately inside each fold because it is part of the pipeline.

In [ ]:
results = []

for name, model in models.items():

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring={
            "r2": "r2",
            "mae": "neg_mean_absolute_error",
            "mse": "neg_mean_squared_error"
        },
        n_jobs=-1
    )

    r2_scores = scores["test_r2"]
    mae_scores = -scores["test_mae"]
    rmse_scores = np.sqrt(-scores["test_mse"])

    results.append({
        "Model": name,
        "Mean R²": r2_scores.mean(),
        "R² Std": r2_scores.std(),
        "Mean MAE": mae_scores.mean(),
        "Mean RMSE": rmse_scores.mean()
    })

results_df = pd.DataFrame(results).sort_values(
    by="Mean R²",
    ascending=False
).reset_index(drop=True)

display(results_df.round(3))

## 11. Recorded Results from the Original Notebook

The original notebook produced the following 5-fold results:

| Model | Mean R² | R² Std | Mean MAE | Mean RMSE |
|---|---:|---:|---:|---:|
| **Random Forest (100 trees)** | **0.890** | 0.034 | **0.957** | **1.499** |
| Tuned Random Forest | 0.821 | 0.034 | 1.344 | 1.898 |
| Tuned Decision Tree | 0.808 | 0.055 | 1.217 | 1.954 |
| Linear Regression | 0.804 | 0.032 | 1.393 | 1.987 |
| Decision Tree | 0.792 | 0.084 | 1.070 | 2.012 |

The 100-tree Random Forest was selected because it had the highest mean R² and the lowest mean MAE/RMSE among the compared models.

In [ ]:
plt.figure(figsize=(10, 6))
plot_df = results_df.sort_values("Mean R²")

plt.barh(plot_df["Model"], plot_df["Mean R²"])
plt.xlabel("Mean R²")
plt.ylabel("Model")
plt.title("5-Fold Cross-Validation Model Comparison")
plt.xlim(0, 1)
plt.tight_layout()
plt.show()

## 12. Train the Final Model

The selected model is a Random Forest Regressor with 100 trees.

In [ ]:
final_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=100,
        random_state=42
    ))
])

final_model.fit(X_train, y_train)

## 13. Final Test Evaluation

The held-out test set is used only after model selection.

In [ ]:
y_pred = final_model.predict(X_test)

final_r2 = r2_score(y_test, y_pred)
final_mae = mean_absolute_error(y_test, y_pred)
final_rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("Final Model Performance")
print("-----------------------")
print("R²   :", final_r2)
print("MAE  :", final_mae)
print("RMSE :", final_rmse)

### Final Test Results from the Original Notebook

- **R² = 0.803**
- **MAE = 1.209**
- **RMSE = 2.009**

An R² of about 0.80 means the model explains a large portion of the variation in final grades on this held-out test set. The MAE indicates that the predictions are off by about 1.21 grade points on average.

In [ ]:
prediction_comparison = pd.DataFrame({
    "Actual G3": y_test.values,
    "Predicted G3": np.round(y_pred, 2)
})

display(prediction_comparison.head(10))

In [ ]:
plt.figure(figsize=(7, 6))
plt.scatter(y_test, y_pred)

plt.plot([0, 20], [0, 20], linestyle="--")
plt.xlabel("Actual G3")
plt.ylabel("Predicted G3")
plt.title("Actual vs Predicted Final Grade")
plt.tight_layout()
plt.show()

## 14. Feature Importance

Random Forest can estimate how much each transformed feature contributes to reducing prediction error.

Because categorical features are one-hot encoded, the feature names include the original feature and category.

In [ ]:
feature_names = final_model.named_steps["preprocessor"].get_feature_names_out()
importances = final_model.named_steps["model"].feature_importances_

feature_importance = pd.Series(
    importances,
    index=feature_names
).sort_values(ascending=False)

display(feature_importance.head(10))

In [ ]:
top_features = feature_importance[feature_importance > 0].head(10).sort_values()

plt.figure(figsize=(9, 5))
plt.barh(top_features.index, top_features.values)
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Top Random Forest Feature Importances")
plt.tight_layout()
plt.show()

### Feature Importance Observation

In the original notebook's tuned Decision Tree analysis, `G2` had an importance of approximately **0.825**, followed by `absences` at approximately **0.175**.

This reinforces the EDA observation that previous academic performance is highly informative for predicting `G3`.

## 15. Prediction for a New Student

The final model expects the following 32 input features.

In [ ]:
required_features = [
    "school", "sex", "age", "address", "famsize", "Pstatus",
    "Medu", "Fedu", "Mjob", "Fjob", "reason", "guardian",
    "traveltime", "studytime", "failures", "schoolsup", "famsup",
    "paid", "activities", "nursery", "higher", "internet",
    "romantic", "famrel", "freetime", "goout", "Dalc", "Walc",
    "health", "absences", "G1", "G2"
]

In [ ]:
def predict_student(student_data):
    # Validate required and extra features.
    missing_features = [
        feature for feature in required_features
        if feature not in student_data
    ]

    extra_features = [
        feature for feature in student_data
        if feature not in required_features
    ]

    if missing_features:
        print("Error: Missing features:")
        print(missing_features)
        return None

    if extra_features:
        print("Error: Unknown features:")
        print(extra_features)
        return None

    # Keep the same feature order expected by the model.
    student_df = pd.DataFrame(
        [[student_data[feature] for feature in required_features]],
        columns=required_features
    )

    # Check for missing values.
    if student_df.isnull().any().any():
        print("Error: Student data contains missing values.")
        return None

    # Check numerical values are valid numbers.
    numeric_input = student_df[numerical_cols]
    if not np.isfinite(numeric_input.to_numpy(dtype=float)).all():
        print("Error: Numerical inputs must contain finite values.")
        return None

    prediction = final_model.predict(student_df)[0]

    print("Student Performance Prediction")
    print("------------------------------")
    print("Predicted Final Grade (G3):", round(prediction, 2))

    return round(prediction, 2)

In [ ]:
student = {
    "school": "GP",
    "sex": "F",
    "age": 17,
    "address": "U",
    "famsize": "GT3",
    "Pstatus": "A",
    "Medu": 4,
    "Fedu": 4,
    "Mjob": "teacher",
    "Fjob": "other",
    "reason": "course",
    "guardian": "mother",
    "traveltime": 2,
    "studytime": 2,
    "failures": 0,
    "schoolsup": "no",
    "famsup": "yes",
    "paid": "no",
    "activities": "yes",
    "nursery": "yes",
    "higher": "yes",
    "internet": "yes",
    "romantic": "no",
    "famrel": 4,
    "freetime": 3,
    "goout": 3,
    "Dalc": 1,
    "Walc": 1,
    "health": 3,
    "absences": 2,
    "G1": 10,
    "G2": 8
}

predicted_grade = predict_student(student)

## 16. Save the Final Model

This saves the complete pipeline, including preprocessing and the Random Forest model, so the same preprocessing is automatically used later.

In [ ]:
import joblib

joblib.dump(final_model, "../models/final_model.pkl")
print("Model saved to ../models/final_model.pkl")

## 17. Conclusion

This project developed an end-to-end machine learning workflow for predicting a student's final grade (`G3`) from 32 input features.

The workflow included:

- Dataset inspection
- Exploratory Data Analysis
- Correlation analysis
- Train-test splitting
- Categorical preprocessing using One-Hot Encoding
- Regression model comparison
- Five-fold cross-validation
- Final model selection
- Test-set evaluation
- Feature importance analysis
- Reusable prediction function
- Model saving

Five regression approaches were compared. The **Random Forest Regressor with 100 trees** achieved the strongest cross-validation performance in the original notebook, with a mean **R² of 0.890**, mean **MAE of 0.957**, and mean **RMSE of 1.499**.

On the held-out test set, the final model achieved **R² = 0.803**, **MAE = 1.209**, and **RMSE = 2.009**.

A key observation is that earlier grades, especially `G2`, are highly informative when predicting `G3`. Therefore, this model is designed for situations where earlier-period grades are already available.

Overall, the project demonstrates a complete practical machine learning workflow from raw student data to a reusable prediction system.

## 18. Project Structure

```text
Student_Performance_Prediction/
│
├── data/
│   └── student-mat.csv
│
├── notebooks/
│   └── Student_Performance_Prediction_Final.ipynb
│
├── models/
│   └── final_model.pkl
│
├── images/
│   └── model_comparison.png
│
├── README.md
├── requirements.txt
└── .gitignore
```